# Data Wrangling and Feature Engineering
This notebook cleans the course-provided SpaceX dataset, documents missing values, creates a binary landing-success target and encodes categorical predictors. It deliberately reports observed results rather than hard-coding expected counts.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(DATA_URL)
print("Initial shape:", df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))

Initial shape: (101, 10)


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


,missing
Date,0
Time (UTC),0
Booster_Version,0
Launch_Site,0
Payload,0
PAYLOAD_MASS__KG_,0
Orbit,0
Customer,0
Mission_Outcome,0
Landing_Outcome,0


In [6]:

# Standardize dataset column names
df.columns = [c.strip() for c in df.columns]


column_mapping = {
    "Flight_Number": "FlightNumber",
    "FlightNumber": "FlightNumber",
    "PAYLOAD_MASS__KG_": "PayloadMass",
    "Payload_Mass": "PayloadMass",
    "Launch_Site": "LaunchSite",
}

df = df.rename(columns=column_mapping)

# Some versions of the dataset use different flight-number names.
if "FlightNumber" not in df.columns:
    if "Flight_Number" in df.columns:
        df["FlightNumber"] = df["Flight_Number"]
    else:
        # Create a sequential identifier if no flight number is supplied.
        df["FlightNumber"] = range(1, len(df) + 1)

required = [
    "FlightNumber",
    "PayloadMass",
    "Orbit",
    "LaunchSite",
    "Landing_Outcome",
]

missing_cols = [c for c in required if c not in df.columns]

if missing_cols:
    raise KeyError(
        f"Missing required columns: {missing_cols}. "
        f"Available columns: {df.columns.tolist()}"
    )

print("Column names standardized successfully!")
print("Dataset shape:", df.shape)
display(df.head())


Column names standardized successfully!
Dataset shape: (101, 11)


,Date,Time (UTC),Booster_Version,LaunchSite,Payload,PayloadMass,Orbit,Customer,Mission_Outcome,Landing_Outcome,FlightNumber
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute),1
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),2
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt,3
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt,4
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt,5


In [7]:
# Handle payload missing values using the median, which is less sensitive to outliers than the mean.
df["PayloadMass"] = pd.to_numeric(df["PayloadMass"], errors="coerce")
payload_median = df["PayloadMass"].median()
df["PayloadMass"] = df["PayloadMass"].fillna(payload_median)

# Define success from landing outcome labels. 'Success' outcomes are positive;
# failures, no-attempt and controlled/uncontrolled ocean outcomes are not counted as successful landings.
df["Class"] = df["Landing_Outcome"].astype(str).str.startswith("Success").astype(int)
print("Missing payload values after imputation:", int(df["PayloadMass"].isna().sum()))
display(df["Class"].value_counts(dropna=False).rename("count").to_frame())

Missing payload values after imputation: 0


,count
Class,
1,61
0,40


In [8]:
# Select useful predictors available in this dataset.
numeric_candidates = [c for c in ["FlightNumber", "PayloadMass"] if c in df.columns]
categorical_candidates = [c for c in ["Orbit", "LaunchSite", "LandingPad", "Serial", "Booster_Version"] if c in df.columns]
features = df[numeric_candidates + categorical_candidates].copy()
for col in numeric_candidates:
    features[col] = pd.to_numeric(features[col], errors="coerce")
    features[col] = features[col].fillna(features[col].median())
for col in categorical_candidates:
    features[col] = features[col].fillna("Unknown").astype(str)

X = pd.get_dummies(features, columns=categorical_candidates, drop_first=False, dtype=int)
y = df["Class"].astype(int)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())

Feature matrix shape: (101, 111)
Target shape: (101,)


,FlightNumber,PayloadMass,Orbit_GTO,Orbit_HEO,Orbit_LEO,Orbit_LEO (ISS),Orbit_MEO,Orbit_Polar LEO,Orbit_SSO,Orbit_Sub-orbital,...,Booster_Version_F9 v1.1 B1003,Booster_Version_F9 v1.1 B1010,Booster_Version_F9 v1.1 B1011,Booster_Version_F9 v1.1 B1012,Booster_Version_F9 v1.1 B1013,Booster_Version_F9 v1.1 B1014,Booster_Version_F9 v1.1 B1015,Booster_Version_F9 v1.1 B1016,Booster_Version_F9 v1.1 B1017,Booster_Version_F9 v1.1 B1018
0,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,525,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,500,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,677,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
df.to_csv("spacex_cleaned.csv", index=False)
X.to_csv("spacex_features.csv", index=False)
y.to_frame("Class").to_csv("spacex_target.csv", index=False)
print("Saved spacex_cleaned.csv, spacex_features.csv, and spacex_target.csv")

Saved spacex_cleaned.csv, spacex_features.csv, and spacex_target.csv


## Data-wrangling notes
- Missing payload mass is imputed with the median.
- Landing outcomes beginning with `Success` are encoded as 1; other outcomes are encoded as 0.
- Categorical predictors are one-hot encoded.
- The feature matrix and target are saved separately for reproducibility.
- Review the dataset's outcome semantics before interpreting the target as a universal definition of landing success.